In [129]:
import torch
import torch.nn as nn
import tools_diff as tools_diff
 # type: ignore
device = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)
print("Device:", device)

CUDA available: True
CUDA Version: 12.1
Device: cuda


# PyTorch Autograd — Comprendre le moteur interne de différentiation

PyTorch utilise un **graphe computationnel dynamique** :

Chaque opération tensorielle :

- crée un nouveau noeud du graphe,
- mémorise la dérivée locale,
- prépare automatiquement la propagation inverse.

---

## Deux attributs fondamentaux d’un tenseur

### `.grad_fn`

Indique **quelle opération a créé ce tenseur**. Exemples : `PowBackward0`

Cela représente le **noeud backward associé dans le graphe**.

---

### `.is_leaf`

Indique si le tenseur est une **feuille du graphe** :

Un tenseur leaf :

- est créé directement par l’utilisateur,
- peut recevoir un gradient dans `.grad`.

Un tenseur non-leaf :

- est produit par une opération,
- ne stocke pas son gradient sauf si `retain_grad()`.

---

d'autres ressources :
- calcul sur graphe https://colah.github.io/posts/2015-08-Backprop/
- codage manuel en python 
https://vmartin.fr/comprendre-la-differenciation-automatique-en-30-lignes-de-python-fr.html

In [130]:
# Explorer le graphe interne

def f(x):
    return torch.sqrt(x)

x = torch.tensor(3.0, requires_grad=True)
a = f(x)
b = a + 1
y = b**2
print("valeur de x au départ =", x.item())
print("Parents de y=(sqrt(x) +1)²  :", y.grad_fn)
print("Parent de b :", y.grad_fn.next_functions[0][0])
print("Parent de a :", y.grad_fn.next_functions[0][0].next_functions[0][0])
print("Parent de x :", y.grad_fn.next_functions[0][0].next_functions[0][0].next_functions[0][0])

#  x --> a --> b --> y
print("a.is_leaf =", a.is_leaf)
print("x.is_leaf =", x.is_leaf)

valeur de x au départ = 3.0
Parents de y=(sqrt(x) +1)²  : <PowBackward0 object at 0x7fa610174520>
Parent de b : <AddBackward0 object at 0x7fa6101759c0>
Parent de a : <SqrtBackward0 object at 0x7fa6101759c0>
Parent de x : <AccumulateGrad object at 0x7fa610174520>
a.is_leaf = False
x.is_leaf = True


In [131]:
# Enregistrement des hooks pour intercepter les gradients
y.register_hook(lambda g: print("\nGradient pour y:", g)) # hook reçoit le gradient exact qui traverse ce noeud pendant backward
b.register_hook(lambda g: print("Gradient pour b:", g))
a.register_hook(lambda g: print("Gradient pour a:", g))
# Backward complet
y.backward()
print("Gradient final dy/dx accumulé dans 'leaf node' x:", x.grad.item())


Gradient pour y: tensor(1.)
Gradient pour b: tensor(5.4641)
Gradient pour a: tensor(5.4641)
Gradient final dy/dx accumulé dans 'leaf node' x: 1.577350378036499


In [132]:

x = torch.tensor([1.0,2.0,3.0,4.0], requires_grad=True)
y = torch.rand(4,requires_grad=True)
z = x+y
print("z.grad_fn ", z.grad_fn) 
print("z.grad_fn.next_functions ", *z.grad_fn.next_functions,sep='\n')  
 # <AddBackward0 object at
a,b = torch.split(x,2)
c = a  + b
print("c.grad_fn.next_functions ", *c.grad_fn.next_functions,sep='\n')  
t = x +1 
print("t.grad_fn.next_functions ", *t.grad_fn.next_functions,sep='\n') 

z.grad_fn  <AddBackward0 object at 0x7fa61012f100>
z.grad_fn.next_functions 
(<AccumulateGrad object at 0x7fa61012d9f0>, 0)
(<AccumulateGrad object at 0x7fa61012dd20>, 0)
c.grad_fn.next_functions 
(<SplitBackward0 object at 0x7fa61012c6d0>, 0)
(<SplitBackward0 object at 0x7fa61012c6d0>, 1)
t.grad_fn.next_functions 
(<AccumulateGrad object at 0x7fa61012f580>, 0)
(None, 0)


| Élément     | Valeurs possibles grad_fn.next_functions                                                      | Signification                                                       |
| ----------- | ---------------------------------------------------------------------- | ------------------------------------------------------------------- |
| `parent_fn` | backward node (`AddBackward0`, `MulBackward0`, `SplitBackward0`, etc.) | parent différentiable                                               |
| `parent_fn` | `AccumulateGrad`                                                       | tensor feuille (`leaf tensor`)                                      |
| `parent_fn` | `None`                                                                 | pas de gradient (constante / detach)                                |
| `idx`       | `0`                                                                    | sortie unique du parent                                             |
| `idx`       | `1, 2, ...`                                                            | sortie n°1, n°2… d’un parent multi-sorties (`split`, `chunk`, etc.) |


# II. torch.autograd

In [133]:
# Custom autograd Function
class SquareFunction(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x):
        # ctx = contexte qui permet de stocker des infos pour le backward
        ctx.save_for_backward(x)   # sauvegarde x pour l'utiliser lors du backward
        return x**2                # calcul direct de y = x^2

    @staticmethod
    def backward(ctx, grad_output):
        # récupère x sauvegardé dans forward
        x, = ctx.saved_tensors
        # dérivée locale de x^2 = 2*x, puis multiplication par grad_output
        # grad_output = dL/dy, donc grad_x = dL/dx
        grad_x = grad_output * 2 * x
        return grad_x

In [134]:
# Utilisation custom backward
target = torch.tensor(5.0)

x = torch.tensor(3.0, requires_grad=True, device=device)

y = SquareFunction.apply(x)

L = (y - target)**2

dL_dy = 2*(y-target)

y.backward(dL_dy) # grad_output = dL_dy
# Ici on ne backward pas la loss,
# on backward directement y,
# en injectant le gradient externe dL/dy

print("x =", x.item(), "y =", y.item())
print("Loss =", L.item(),"Gradient =", x.grad.item())

x = 3.0 y = 9.0
Loss = 16.0 Gradient = 48.0


In [135]:
# Profiler autograd
x = torch.tensor(3.0, requires_grad=True)
# x.register_hook(lambda g: print("Gradient intercepté :", g))

y = SquareFunction.apply(x)

with torch.autograd.profiler.profile() as prof:
    y.backward()

print(prof.key_averages())

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                        aten::ones_like         4.66%       4.617us        17.93%      17.760us      17.760us             1  
                                       aten::empty_like         5.65%       5.592us        11.69%      11.575us      11.575us             1  
                                    aten::empty_strided         6.79%       6.722us         6.79%       6.722us       3.361us             2  
                                            aten::fill_         1.58%       1.568us         1.58%       1.568us       1.568us             1  
autogr

In [136]:
# torch.autograd.grad
x = torch.tensor(3.0, requires_grad=True)
y = SquareFunction.apply(x)
grad_x = torch.autograd.grad(y, x)

print("torch.autograd de y par rapport à x:", grad_x[0].item())
t = torch.tensor(3.0, requires_grad=True)

z = x + 2*t 
z.backward()
print("Gradient de z par rapport à x:", x.grad.item())
print("Gradient de z par rapport à t:", t.grad.item())



torch.autograd de y par rapport à x: 6.0
Gradient de z par rapport à x: 1.0
Gradient de z par rapport à t: 2.0


| Méthode                 | Calcule gradient | Stocke dans `.grad` | Accumule dans toutes les feuilles | Retourne valeur |
| ----------------------- | ---------------- | ------------------- | -------- | --------------- |
| `backward(dL)`            | oui              | oui                 | oui      | non             |
| `torch.autograd.grad(y,x)` | oui              | non                 | non      | oui             |


In [137]:
# Accumulation gradients et stockage du graphe avec retain_graph
x = torch.tensor(3.0, requires_grad=True)

y = x**2

y.backward(retain_graph=True) # si pas cette option , erreur à la 2eme car backward détruit normalement le graphe pour libérer mémoire 
# retain_graph=True empêche cette destruction
print("1er backward:", x.grad.item())
y.backward()
print("2eme backward:", x.grad.item()) # accumulation du gradient : 2*x + 2*x = 4*x = 12.0
# retain_graph=True peut s'appliquer à torch.autograd.grad car par défaut cette fonction consomme aussi le graphe après le calcul du gradient


1er backward: 6.0
2eme backward: 12.0


In [138]:
# Reset gradient
x = torch.tensor(3., requires_grad=True)

y = x**2
y.backward()
print(x.grad)
x.grad = None  # utilisé par défaut dans optimizer.zero_grad(set_to_none=True), pas besoin de  mémoire

x = torch.tensor(3., requires_grad=True)
y = x**2
y.backward()
x.grad.zero_() # on garde le tensor gradient existant et on remplit avec 0
print(x.grad)


tensor(6.)
tensor(0.)


| Méthode          | Effet                            | Mémoire           | Recommandation |
| ---------------- | -------------------------------- | ----------------- | -------------- |
| `x.grad = None`  | supprime le gradient             | libère / réalloue | ✅ recommandé   |
| `x.grad.zero_()` | garde tensor gradient et met à 0 | conserve mémoire  | parfois utile  |


In [139]:
# retain_grad()  Par défaut dans PyTorch :  seuls les leaf tensors gardent .grad   les intermédiaires perdent leur gradient
# leaf tensor
x = torch.tensor(2.0, requires_grad=True)
# tenseur intermédiaire
y = x**2
# sans retain_grad(), y.grad resterait None
y.retain_grad()
z = y**3

z.backward()
print("x.grad =", x.grad.item())   # dz/dx
print("y.grad =", y.grad.item())   # dz/dy

x.grad = 192.0
y.grad = 48.0


In [140]:
# create_graph=True pour calculer des gradients d'ordre supérieur
x = torch.tensor(4.0, requires_grad=True)

y = x**3

grad1 = torch.autograd.grad(y, x)[0]

print(grad1)
print(grad1.requires_grad)

# # Parce que PyTorch a dérivé 3x² puis évalué en x=4 : 48

x = torch.tensor(4.0, requires_grad=True)
y = x**3
grad1 = torch.autograd.grad(y, x, create_graph=True)[0] # je veux le gradient ET son graphe pour pouvoir le redériver
print("grad d'ordre 1 :", grad1)
print(grad1.requires_grad)

grad2 = torch.autograd.grad(grad1, x)[0] 

print("grad d'ordre 2 :", grad2) # la dérive 6x et directement évalué en 4: = 24


tensor(48.)
False
grad d'ordre 1 : tensor(48., grad_fn=<MulBackward0>)
True
grad d'ordre 2 : tensor(24.)


In [141]:
# Hessien
# Fonction scalaire : f(x) = x^4
def f(x):
    return x**4

# x sans requires_grad nécessaire ici :
# torch.autograd.functional gère lui-même les gradients internes
x = torch.tensor(2.0)

# Valeur de la fonction
y = f(x)

# Hessian = dérivée seconde
H = torch.autograd.functional.hessian(f, x)

print("f(x) =", y.item())
print("Hessian =", H.item())

f(x) = 16.0
Hessian = 48.0


In [142]:
# Jaccobian 

def g(x):
    return torch.stack([x**2, x**3])

x = torch.tensor(2.0)
# Valeur de la fonction
y = g(x)
# Jacobian
J = torch.autograd.functional.jacobian(g, x)

print("g(x) =", y)
print("Jacobian =", J)

g(x) = tensor([4., 8.])
Jacobian = tensor([ 4., 12.])


In [143]:
x = torch.tensor(3.0, requires_grad=True)

y = x**2

# z partage la valeur de y mais sans historique gradient
z = y.detach()    

print(f"y={y} et y.requires_grad =", y.requires_grad)
print(f"z={z} et z.requires_grad =", z.requires_grad)

# detach coupe totalement le gradient :
try:
    z.backward()
except:
    print("Impossible : z n'a plus de graphe")

y=9.0 et y.requires_grad = True
z=9.0 et z.requires_grad = False
Impossible : z n'a plus de graphe


In [144]:
x = torch.tensor(3.0, requires_grad=True)

with torch.no_grad(): # tout ce qui est calculé dans ce bloc n'aura pas de suivi de gradient
    y = x**2

print("y.requires_grad =", y.requires_grad)

y.requires_grad = False


| méthode   | moment de coupure |
| --------- | ----------------- |
| detach()  | après calcul      |
| no_grad() | pendant calcul    |


In [145]:

# Couche linéaire : y = wx + b
layer = nn.Linear(1, 1)

# entrée batch 1 × 1
x = torch.tensor([[2.0]])

# forward
y = layer(x)

print("weight =", layer.weight.data)
print("bias =", layer.bias.data)
print("output y =", y)

weight = tensor([[-0.8715]])
bias = tensor([0.3207])
output y = tensor([[-1.4222]], grad_fn=<AddmmBackward0>)


In [146]:
# backward auto
target = torch.tensor([[6.0]])

# perte MSE
loss = nn.MSELoss()(y, target)

# remise à zéro gradients
layer.zero_grad()

# backward
loss.backward()

print("loss =", loss.item())
print("grad weight =", layer.weight.grad) # input grand → gradient poids grand
print("grad bias =", layer.bias.grad)

loss = 55.08956527709961
grad weight = tensor([[-29.6889]])
grad bias = tensor([-14.8445])


In [147]:
# gradient perte par rapport à sortie L=(y−target)² 
dL_dy = 2 * (y - target)

# gradient sortie par rapport au poids
dy_dw = x # w les poids sont multipliés par x, donc la dérivée locale de y par rapport à w est x

# gradient manuel
dL_dw = dL_dy * dy_dw

print("dL/dy =", dL_dy.item())
print("dy/dw =", dy_dw.item())
print("dL/dw =", dL_dw.item()) # retrouve  layer.weight.grad

dL/dy = -14.84446907043457
dy/dw = 2.0
dL/dw = -29.68893814086914


In [148]:
# backward explicite custom linear
class LinearFunction(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x, weight, bias):
        # on garde pour backward
        ctx.save_for_backward(x, weight, bias)
        # y = wx + b
        return weight * x + bias

    @staticmethod
    def backward(ctx, grad_output): # grad_output = gradient venant de la loss :dL/dy
        x, weight, bias = ctx.saved_tensors
        # dL/dx = dL/dy * dy/dx
        grad_x = grad_output * weight
        # dL/dw = dL/dy * dy/dw
        grad_weight = grad_output * x
        # dL/db = dL/dy * 1
        grad_bias = grad_output 
        return grad_x, grad_weight, grad_bias

In [149]:
# couche custom complète
class CustomLinearLayer(nn.Module):

    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.tensor(1.0)) # poids initialisé à 1.0
        self.bias = nn.Parameter(torch.tensor(0.0))

    def forward(self,x):
        return LinearFunction.apply(x,self.weight,self.bias)
    
layer = CustomLinearLayer()


In [150]:
# test training manuel
x = torch.tensor(2.0)
target = torch.tensor(4.0)

for idx in range(10):
    layer.zero_grad()
    pred = layer(x)
    loss = (pred - target)**2
    loss.backward()


    # update manuel
    lr = 0.01
    with torch.no_grad(): # sinon l'update lui-même entrerait dans le graphe autograd
        layer.weight -= lr * layer.weight.grad
        layer.bias -= lr * layer.bias.grad

    pred = layer(x)
    loss = (pred - target)**2

    print(f" iter {idx}/10 prediction = {round(pred.item(),3)}, with loss = {round(loss.item(),3)}")

 iter 0/10 prediction = 2.2, with loss = 3.24
 iter 1/10 prediction = 2.38, with loss = 2.624
 iter 2/10 prediction = 2.542, with loss = 2.126
 iter 3/10 prediction = 2.688, with loss = 1.722
 iter 4/10 prediction = 2.819, with loss = 1.395
 iter 5/10 prediction = 2.937, with loss = 1.13
 iter 6/10 prediction = 3.043, with loss = 0.915
 iter 7/10 prediction = 3.139, with loss = 0.741
 iter 8/10 prediction = 3.225, with loss = 0.6
 iter 9/10 prediction = 3.303, with loss = 0.486
